# 02 - Data Cleaning & Preprocessing

## Digikala Content Analytics & AI Insight Platform


## Business Context

Raw e-commerce data usually contains:

- Missing information
- Duplicate records
- Inconsistent formats
- Unprocessed customer text

Data cleaning is a critical step to ensure reliable analytics
and accurate AI models.


## Objective

The goal of this notebook is to:

- Assess raw data quality
- Clean product and review datasets
- Handle missing values
- Remove duplicated records
- Normalize text features
- Prepare datasets for analytics and machine learning


## Workflow


Raw Data

↓

Quality Assessment

↓

Cleaning

↓

Feature Preparation

↓

Processed Dataset



# 1. Import Libraries


Libraries used for:

- Data manipulation
- Text processing
- Visualization
- File management


In [1]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


import os
import re


import warnings

warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    None
)


print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Project Configuration


Define input and output paths.

Raw data will remain unchanged.
Cleaned datasets will be stored separately.

In [2]:
RAW_PATH = "../data/raw/"

PROCESSED_PATH = "../data/processed/"


PRODUCT_FILE = (
    RAW_PATH +
    "digikala-products.csv"
)


COMMENTS_FILE = (
    RAW_PATH +
    "digikala-comments.csv"
)


os.makedirs(
    PROCESSED_PATH,
    exist_ok=True
)


print("Paths configured.")

Paths configured.


# 3. Load Data


Because the comments dataset is very large,
we initially process a sample.

Full-scale processing can later be implemented
using chunk-based techniques.

In [3]:
products = pd.read_csv(
    PRODUCT_FILE
)


comments = pd.read_csv(
    COMMENTS_FILE,
    nrows=200000
)


print(
    "Products:",
    products.shape
)


print(
    "Comments:",
    comments.shape
)

Products: (1283496, 12)
Comments: (200000, 15)


# 4. Initial Data Quality Assessment


Before cleaning, we analyze:

- Missing values
- Duplicate rows
- Data types
- Invalid records


In [4]:
def data_quality_report(df):

    report = pd.DataFrame({

        "Data_Type": df.dtypes,

        "Missing_Count": df.isnull().sum(),

        "Missing_Percentage":
        (
            df.isnull().mean()*100
        ),

        "Unique_Values":
        df.nunique()

    })


    return report.sort_values(
        "Missing_Percentage",
        ascending=False
    )


products_quality = data_quality_report(products)


products_quality.head(20)

,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
Category2,object,210108,16.369977,601
Seller,object,223,0.017374,18584
id,int64,0,0.000000,948352
title_fa,object,0,0.000000,907314
Rate,int64,0,0.000000,51
Rate_cnt,int64,0,0.000000,3123
Category1,object,0,0.000000,228
Brand,object,0,0.000000,8961
Price,int64,0,0.000000,46810
Is_Fake,bool,0,0.000000,2


In [5]:
comments_quality = data_quality_report(comments)

comments_quality.head(20)

,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
true_to_size_rate,object,197390,98.6950,5
disadvantages,object,187757,93.8785,7229
advantages,object,178025,89.0125,16304
title,object,95345,47.6725,39224
recommendation_status,object,28612,14.3060,3
seller_title,object,8721,4.3605,7259
seller_code,object,8721,4.3605,7257
body,object,14,0.0070,154038
id,int64,0,0.0000,199993
created_at,object,0,0.0000,2074


# 5. Duplicate Analysis


Duplicate records may cause:

- Incorrect KPIs
- Biased models
- Wrong business decisions



In [6]:
print(
    "Product duplicates:",
    products.duplicated().sum()
)


print(
    "Comment duplicates:",
    comments.duplicated().sum()
)

Product duplicates: 323129
Comment duplicates: 3


# 6. Removing Duplicate Records

Duplicate rows are removed to improve
data reliability.


In [7]:
products_clean = (
    products
    .drop_duplicates()
    .copy()
)


comments_clean = (
    comments
    .drop_duplicates()
    .copy()
)


print(products_clean.shape)

print(comments_clean.shape)

(960367, 12)
(199997, 15)


# 7. Missing Value Handling


Not all missing values should be removed.

Business logic determines the correct strategy.

Examples:

- Missing text → empty string
- Missing categorical values → Unknown
- Missing numerical values → Median



In [8]:
products_clean.isnull().sum().head(20)

id                           0
title_fa                     0
Rate                         0
Rate_cnt                     0
Category1                    0
Category2               181890
Brand                        0
Price                        0
Seller                     198
Is_Fake                      0
min_price_last_month         0
sub_category                 0
dtype: int64

In [9]:
comments_clean.isnull().sum().head(20)

id                            0
title                     95342
body                         14
created_at                    0
rate                          0
recommendation_status     28611
is_buyer                      0
product_id                    0
advantages               178022
disadvantages            187754
likes                         0
dislikes                      0
seller_title               8721
seller_code                8721
true_to_size_rate        197387
dtype: int64

# 8. Text Cleaning Preparation


Customer reviews are used later for:

- Sentiment Analysis
- Topic Extraction
- AI Summarization


We create a basic text normalization function.


In [10]:
def clean_persian_text(text):

    if pd.isna(text):
        return ""


    text = str(text)


    # Remove extra spaces

    text = re.sub(
        r'\s+',
        ' ',
        text
    )


    # Remove unwanted characters

    text = re.sub(
        r'[^\w\sآ-ی]',
        '',
        text
    )


    return text.strip()

In [11]:
# Apply text cleaning on comments


text_columns = [
    col for col in comments_clean.columns
    if "comment" in col.lower()
]


text_columns

[]

In [12]:
for col in text_columns:

    comments_clean[col] = (
        comments_clean[col]
        .apply(clean_persian_text)
    )


comments_clean.head()

,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,NaN,NaN,0,0,دیجی‌کالا,5A52N,NaN
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.0,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],1,0,دیجی‌کالا,5A52N,NaN
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.0,recommended,True,3331597,NaN,NaN,0,0,آرالیا بیوتی,ADM47,NaN
3,18628562,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,28 اسفند 1399,0.0,recommended,True,3331329,NaN,NaN,0,0,اینجاست آ,9ZMCZ,NaN
4,53301258,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,12 شهریور 1402,3.0,recommended,True,3255700,NaN,NaN,0,0,گالری آرایشی به سیما,CDWHA,NaN


# 9. Save Processed Data


Clean datasets are saved separately
to preserve reproducibility.


Output:


data/

└── processed/

    ├── products_clean.csv

    └── comments_clean.csv


In [14]:
products_clean.to_csv(
    PROCESSED_PATH +
    "products_clean.csv",
    index=False
)


comments_clean.to_csv(
    PROCESSED_PATH +
    "comments_clean.csv",
    index=False
)


print(
    "Processed files saved successfully."
)

Processed files saved successfully.


# 10. Summary


## Completed Tasks

✔ Loaded raw datasets

✔ Evaluated data quality

✔ Removed duplicated records

✔ Prepared missing value handling

✔ Normalized review text

✔ Saved processed datasets


## Next Steps


The next notebook focuses on:


# Exploratory Data Analysis (EDA)


We will analyze:

- Product distribution
- Category performance
- Customer ratings
- Review patterns
- Initial business insights
